# 第 5 章 · 动作模仿：从哪开始练、什么时候喊停、难的地方多练

配套讲义：仓库里 `robot-motion-rl/notes/lessons/05-imitation-sampling.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（片段长度、各段回报是编的，50/40/10 和温度 25 取自 SuperMimic 附录 D.1）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 RSI | 窗口 / 片段长 | `rsi_reach_probability(...)` | 从头 0；RSI 0.2 |
| 第四步 softmax | $\text{softmax}(-25G/G_{max})$ | `reset_distribution(g)[1]` | [0, 0, 1, 0, 0] |
| 第四步 混合 | 0.5 + 0.4 + 0.1 | `reset_distribution(g)[0]` | [0.1, 0.1, 0.6, 0.1, 0.1] |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `softmax`**：softmax：先减最大值防溢出，再 exp、归一化。

In [ ]:
def softmax(x):
    """softmax：先减最大值防溢出，再 exp、归一化。"""
    z = np.exp(np.asarray(x, float) - np.max(x))
    return z / z.sum()

**函数 `reset_distribution`**：SuperMimic 附录 D.1：50% 均匀 + 40% softmax(−temp·G_b/G_max) + 10% 历史权重（不给就用当前 softmax）。

In [ ]:
def reset_distribution(g_over_gmax, past_weight=None, temp=25.0):
    """SuperMimic 附录 D.1：50% 均匀 + 40% softmax(−temp·G_b/G_max) + 10% 历史权重（不给就用当前 softmax）。"""
    g = np.asarray(g_over_gmax, float)
    hard = softmax(-temp * g)
    past = hard if past_weight is None else np.asarray(past_weight, float)
    uniform = np.full_like(g, 1.0 / len(g))
    return 0.5 * uniform + 0.4 * hard + 0.1 * past, hard

**函数 `rsi_reach_probability`**：从片头开始时，策略能撑 survive 秒才可能练到 flip_at 处；RSI 均匀抽起点时练到它的概率（编的简化）。

In [ ]:
def rsi_reach_probability(start_phase_uniform, clip_seconds, flip_at, survive_seconds):
    """从片头开始时，策略能撑 survive 秒才可能练到 flip_at 处；RSI 均匀抽起点时练到它的概率（编的简化）。"""
    if not start_phase_uniform:
        return 1.0 if survive_seconds >= flip_at else 0.0
    window = min(survive_seconds, clip_seconds)
    return min(1.0, window / clip_seconds)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 参考状态初始化 RSI")
print("10 s 片段，空翻在第 6 s，策略只能撑 2 s：从片头开始练到空翻的概率 =", rsi_reach_probability(False, 10, 6, 2))
print("RSI 均匀抽起点，起点落在空翻前 2 s 内的概率 =", rsi_reach_probability(True, 10, 6, 2))

In [ ]:
print("\n第二步 按回报重置")
g = [0.9, 0.9, 0.2, 0.8, 0.9]
mix, hard = reset_distribution(g)
print("各段 G/G_max =", g)
print("softmax(−25·G/G_max) =", np.round(hard, 3))
print("最终重置分布 =", np.round(mix, 3))

## 3. 代码里才看得到的

- softmax 先减最大值再 exp，防止 $e^{-5}$ 和 $e^{-22.5}$ 这种数相差太大时溢出或下溢。
- 「过去的权重」在真实训练里是历史上 softmax 的累计，例子为了简单直接用当前值；所以 0.4 + 0.1 都押在第 3 段，得 0.6。

## 4. 自己改一改

- 把第 3 段的回报从 0.2 慢慢提到 0.9，看最终分布怎么回到均匀。
- 把温度改成 5、1，看 softmax 怎么从「全押」变成「平均」。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 跳舞照视频，奖励就好写；
> 难处在中间，随机开局才练得到；
> 摔了就喊停，躺着的样本不要；
> 一半均匀保底，四成押最差，一成照顾学会的。